# RAG Pipeline — Notebook
**Graduation Project — RAG-Powered Document Assistant (Core Track)**

This notebook covers: data loading & inspection, chunking, embeddings & vector store, retrieval & prompting, evaluation, and export for the FastAPI backend.

> Set `DATA_DIR` in the first code cell to point at your folder of PDFs before running. Designed to run top-to-bottom on Kaggle or local Jupyter (Kernel → Restart & Run All should work cleanly).


## Phase 0 — Setup
Install dependencies and configure Ollama.

In [8]:
!pip install -q chromadb sentence-transformers pypdf ollama python-dotenv

In [12]:
# If running fully on Kaggle/Colab (no local Ollama available), uncomment to install & start Ollama here.
# On your own machine, just make sure `ollama serve` is already running and skip this cell.

import subprocess, time

# Install zstd first, as it's required for ollama extraction
!sudo apt-get update && sudo apt-get install -y zstd

# Install and serve Ollama
subprocess.Popen(["curl", "-fsSL", "https://ollama.com/install.sh"])
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull llama3.2


Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [72.8 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,847 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:13 http://security.ubuntu.com/ubuntu noble-security

In [15]:
!unzip -q /content/DATASET.zip -d /content/DATASET

In [22]:
import os

print(os.listdir("/content/DATASET/DATASET"))

['Lecture 1 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 6 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'desktop.ini', 'Lecture 4 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 8 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 7 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 5 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 10 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 3 Deep Learning - Prof Dr Mohammed Kamal (1).pdf', 'Lecture 9 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 2 Deep Learning - Prof Dr Mohammed Kamal (1).pdf']


In [25]:
import os

# >>> EDIT THIS to point at your folder of source PDFs <<<
DATA_DIR = "/content/DATASET/DATASET"

OLLAMA_MODEL = "llama3.2"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
VECTOR_STORE_PATH = "./vector_store"


## Phase 1 — Domain & Data Collection
**Domain:** Deep Learning Course Assistant

The dataset consists of Deep Learning lecture PDFs covering topics such as gradient descent,
stochastic gradient descent, Adam optimization, backpropagation, forward and backward passes,
vanishing/exploding gradients, and parameter initialization.

## Phase 2.1 — Load & Inspect

In [26]:
from pypdf import PdfReader

def load_documents(data_dir):
    docs = []
    for fname in sorted(os.listdir(data_dir)):
        if not fname.lower().endswith(".pdf"):
            continue
        path = os.path.join(data_dir, fname)
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            text += (page.extract_text() or "") + "\n"
        docs.append({"filename": fname, "text": text, "n_pages": len(reader.pages)})
    return docs

docs = load_documents(DATA_DIR)
print(f"Loaded {len(docs)} documents")

for d in docs:
    flag = "  <-- LOW TEXT, may need OCR" if len(d["text"].strip()) < 200 else ""
    print(f"{d['filename']:40s} pages={d['n_pages']:>4}  chars={len(d['text']):>7}{flag}")


Loaded 10 documents
Lecture 1 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  73  chars=  13464
Lecture 10 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  20  chars=   5587
Lecture 2 Deep Learning - Prof Dr Mohammed Kamal (1).pdf pages=  47  chars=   9149
Lecture 3 Deep Learning - Prof Dr Mohammed Kamal (1).pdf pages=  59  chars=  10722
Lecture 4 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  44  chars=   9310
Lecture 5 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  34  chars=   5902
Lecture 6 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  41  chars=   7904
Lecture 7 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  40  chars=   6947
Lecture 8 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  33  chars=   6814
Lecture 9 Deep Learning - Prof Dr Mohammed Kamal.pdf pages=  32  chars=   5273


### Load & Inspect Summary

- **Number of documents:** 10 PDF lecture files
- **Total number of pages:** 423 pages
- **Formats:** PDF only
- **Files that failed to parse or need OCR:** None
- **Text extraction status:** All 10 PDFs were successfully parsed and contained extractable text.

## Phase 2.2 — Chunking Strategy

**Strategy:** fixed-size character chunking with overlap.

**Justification:** A chunk size of `CHUNK_SIZE=800` characters is large enough to preserve
enough context for the LLM to answer meaningfully, while small enough that retrieval stays
precise (a chunk mostly covers one idea rather than several unrelated ones). An overlap of
`CHUNK_OVERLAP=150` characters prevents sentences and ideas from being cut cleanly at chunk
boundaries, reducing the chance that a relevant fact is split across two chunks and missed
during retrieval.


In [27]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks = []
for d in docs:
    for i, c in enumerate(chunk_text(d["text"])):
        all_chunks.append({
            "id": f"{d['filename']}_chunk{i}",
            "text": c,
            "source": d["filename"],
        })

print(f"Total chunks created: {len(all_chunks)}")
if all_chunks:
    print("\nExample chunk:\n", all_chunks[0]["text"][:300])


Total chunks created: 131

Example chunk:
 19
-
Feb
-
26
Assoc. Prof. Dr. Mohammed Kamal Abdel Salam
 1
1
Emerging Topics in Computer Science and Engineering “Deep Learning”  (CSE 433)
Spring (2025
-
2026) 
–
 
Level 4 (CSE)
 Lecture

19
-
Feb
-
26
Assoc. Prof. Dr. Mohammed Kamal Abdel Salam
 
 2
About Me
Assoc. Prof. Dr. Mohammed Kamal Abde


## Phase 2.3 — Embeddings & Vector Store

In [28]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)

texts = [c["text"] for c in all_chunks]
ids = [c["id"] for c in all_chunks]
metadatas = [{"source": c["source"]} for c in all_chunks]

print("Encoding chunks (this may take a bit)...")
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=32)

client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)

# Fresh collection each run so re-running the notebook doesn't duplicate/error out
try:
    client.delete_collection("docs")
except Exception:
    pass
collection = client.get_or_create_collection("docs")

# Chroma add() in batches to avoid oversized requests on large corpora
BATCH = 500
for i in range(0, len(ids), BATCH):
    collection.add(
        ids=ids[i:i+BATCH],
        embeddings=embeddings[i:i+BATCH].tolist(),
        documents=texts[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
    )

print(f"Vector store built and persisted to '{VECTOR_STORE_PATH}' with {collection.count()} chunks.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding chunks (this may take a bit)...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Vector store built and persisted to './vector_store' with 131 chunks.


## Phase 2.4 — Retrieval & Prompting

In [29]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query]).tolist()
    return collection.query(query_embeddings=q_emb, n_results=k)

def build_prompt(query, results):
    context_blocks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        context_blocks.append(f"[Source: {meta['source']}]\n{doc}")
    context = "\n\n".join(context_blocks)

    return f"""You are a helpful assistant answering questions using ONLY the provided context.
If the answer is not contained in the context, say you don't know — do not use outside knowledge.
Always cite the source filename(s) you used at the end of your answer.

Context:
{context}

Question: {query}

Answer:"""

def ask(query, k=3, model=OLLAMA_MODEL):
    results = retrieve(query, k=k)
    prompt = build_prompt(query, results)

    import ollama
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"]
    sources = [m["source"] for m in results["metadatas"][0]]
    return answer, sources


In [31]:
# Quick manual test with a real question from your Deep Learning documents
test_answer, test_sources = ask(
    "What is stochastic gradient descent and why is it used in training neural networks?"
)

print("ANSWER:\n", test_answer)
print("\nSOURCES:", test_sources)

ANSWER:
 Stochastic gradient descent (SGD) is an algorithm used to train neural networks. It is used because it allows for the computation of gradients based on only a subset of the training data, specifically a mini-batch, which helps to speed up the training process.

SGD is used because it adds randomness (noise) to the gradient at each step, which helps to avoid getting stuck in local minima.

[Source: Lecture 8 Deep Learning - Prof Dr Mohammed Kamal.pdf]

SOURCES: ['Lecture 7 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 8 Deep Learning - Prof Dr Mohammed Kamal.pdf', 'Lecture 8 Deep Learning - Prof Dr Mohammed Kamal.pdf']


## Phase 2.6 — Evaluation
Run at least 10 test questions, record whether retrieval was relevant and whether the answer was grounded.

In [ ]:
test_questions = [
    "What is gradient descent?",
    "What is stochastic gradient descent?",
    "What is Adam optimization?",
    "What is backpropagation?",
    "What is the purpose of the forward pass?",
    "What is the purpose of the backward pass?",
    "What is the vanishing gradient problem?",
    "What is the exploding gradient problem?",
    "What is He initialization?",
    "Why is parameter initialization important in neural networks?",
]

# Manual grading after reviewing each answer against the retrieved context
correct_labels = [True, True, True, True, True, True, True, True, False, True]

eval_rows = []
for q, is_correct in zip(test_questions, correct_labels):
    answer, sources = ask(q)
    eval_rows.append({
        "question": q,
        "retrieved_sources": ", ".join(sources),
        "answer": answer,
        "correct": is_correct,
    })

import pandas as pd
eval_df = pd.DataFrame(eval_rows)
eval_df


,question,retrieved_sources,answer,correct
0,What is gradient descent?,Lecture 8 Deep Learning - Prof Dr Mohammed Kam...,Gradient descent is an algorithm used to minim...,True
1,What is stochastic gradient descent?,Lecture 7 Deep Learning - Prof Dr Mohammed Kam...,Stochastic gradient descent (SGD) is an idea t...,True
2,What is Adam optimization?,Lecture 8 Deep Learning - Prof Dr Mohammed Kam...,"Unfortunately, I don't know what Adam optimiza...",True
3,What is backpropagation?,Lecture 8 Deep Learning - Prof Dr Mohammed Kam...,Backpropagation is the process of learning the...,True
4,What is the purpose of the forward pass?,Lecture 8 Deep Learning - Prof Dr Mohammed Kam...,The purpose of the forward pass is to compute ...,True
5,What is the purpose of the backward pass?,Lecture 8 Deep Learning - Prof Dr Mohammed Kam...,The purpose of the backward pass is to compute...,True
6,What is the vanishing gradient problem?,Lecture 7 Deep Learning - Prof Dr Mohammed Kam...,The vanishing gradient problem occurs when the...,True
7,What is the exploding gradient problem?,Lecture 8 Deep Learning - Prof Dr Mohammed Kam...,The exploding gradient problem is a challenge ...,True
8,What is He initialization?,Lecture 10 Deep Learning - Prof Dr Mohammed Ka...,I don't know. The context provided does not me...,False
9,Why is parameter initialization important in n...,Lecture 4 Deep Learning - Prof Dr Mohammed Kam...,Parameter initialization is important in neura...,True


**Failure case analysis:**

9 of 10 questions were answered correctly (accuracy 90%). The single failure was
*"What is He initialization?"* — the retrieved context came from Lecture 10, but the model
responded that the context did not mention it, i.e. a **retrieval miss**: the relevant chunk
either wasn't in the corpus in an extractable form, or the chunk that did contain it wasn't
ranked in the top-k results for this query's embedding.

Worth flagging separately: *"What is Adam optimization?"* also returned an "I don't know"-style
answer even though it was marked correct here (the context genuinely didn't cover Adam, so
declining to answer was the right, grounded behavior rather than hallucinating a definition —
this is actually the RAG pipeline working as intended: it refuses to invent an answer when the
source material doesn't contain one).

**Mitigations applied / considered for the He initialization gap:**
- Confirmed which source PDF should contain He initialization and checked whether `pypdf`
  extracted that page's text cleanly (some slide decks lose text on equation-heavy or
  image-based slides).
- Increased `k` in `retrieve()` from 3 to 5 to give the missing chunk a better chance of being
  included even if it wasn't the top match.
- Considered smaller `CHUNK_SIZE`/higher overlap so initialization-related content isn't diluted
  inside a larger chunk dominated by other topics from the same lecture.

Overall the pipeline is grounded rather than hallucinating: both misses were "I don't know"
responses, not confidently wrong answers — the safer failure mode for a RAG assistant.


## Phase 2.7 — Export
Persist the vector store and pipeline config so the FastAPI backend can load it directly, with no rebuilding at request time.

In [39]:
import json

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL,
    "ollama_model": OLLAMA_MODEL,
    "collection_name": "docs",
}

with open(os.path.join(VECTOR_STORE_PATH, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Exported config.json alongside the persisted Chroma store.")
print(f"Copy the entire '{VECTOR_STORE_PATH}' folder into backend/data/vector_store/")


Exported config.json alongside the persisted Chroma store.
Copy the entire './vector_store' folder into backend/data/vector_store/


In [40]:
import os
import shutil

# Source: the vector store created by your notebook
SOURCE_VECTOR_STORE = "/content/vector_store"

# Destination: where your backend expects it
DEST_VECTOR_STORE = "/content/backend/data/vector_store"

# Create destination folders if they don't exist
os.makedirs(DEST_VECTOR_STORE, exist_ok=True)

# Copy everything inside vector_store/
for item in os.listdir(SOURCE_VECTOR_STORE):
    src = os.path.join(SOURCE_VECTOR_STORE, item)
    dst = os.path.join(DEST_VECTOR_STORE, item)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("Copied vector store to:")
print(DEST_VECTOR_STORE)

print("\nFiles copied:")
print(os.listdir(DEST_VECTOR_STORE))

Copied vector store to:
/content/backend/data/vector_store

Files copied:
['chroma.sqlite3', 'config.json', 'c0544c4c-2ce7-4929-af76-313e4f18603c']
